1. Load raw dataset

In [0]:
df = spark.table("workspace.default.raw_covid")
display(df)

2. Select + Rename

In [0]:
from pyspark.sql.functions import col, trim, upper

df_covid_rows = df.filter(
    col("_c0").isNotNull()
).withColumn(
    "_c0", trim(col("_c0"))
).filter(
    ~upper(col("_c0")).startswith("SERIE HISTÓRICA")
).filter(
    ~upper(col("_c0")).startswith("ANO ")
).filter(
    upper(col("_c0")) != "MÊS"
).filter(
    upper(col("_c0")) != "TOTAL"
).filter(
    ~upper(col("_c0")).startswith("DADOS ATUALIZADOS")
)

display(df_covid_rows)

df_covid_month = df_covid_rows.withColumnRenamed("_c0", "mes")

display(df_covid_month)

3. Unpivot

In [0]:
from pyspark.sql.functions import lit

df_2020 = df_covid_month.select(
    "mes",
    lit(2020).alias("ano"),
    col("_c1").alias("casos"),
    col("_c2").alias("obitos")
)

df_2021 = df_covid_month.select(
    "mes",
    lit(2021).alias("ano"),
    col("_c3").alias("casos"),
    col("_c4").alias("obitos")
)

df_2022 = df_covid_month.select(
    "mes",
    lit(2022).alias("ano"),
    col("_c5").alias("casos"),
    col("_c6").alias("obitos")
)

df_2023 = df_covid_month.select(
    "mes",
    lit(2023).alias("ano"),
    col("_c7").alias("casos"),
    col("_c8").alias("obitos")
)

df_2024 = df_covid_month.select(
    "mes",
    lit(2024).alias("ano"),
    col("_c9").alias("casos"),
    col("_c10").alias("obitos")
)

df_covid_long = (
    df_2020
    .unionByName(df_2021)
    .unionByName(df_2022)
    .unionByName(df_2023)
    .unionByName(df_2024)
)

display(df_covid_long)

4. Clean numeric fields

In [0]:
from pyspark.sql.functions import regexp_replace

df_covid_clean = (
    df_covid_long
    .withColumn("casos", regexp_replace(col("casos").cast("string"), r"\.", ""))
    .withColumn("obitos", regexp_replace(col("obitos").cast("string"), r"\.", ""))
    .withColumn("casos", col("casos").cast("int"))
    .withColumn("obitos", col("obitos").cast("int"))
)

display(df_covid_clean)

5. Add disease column

In [0]:
df_covid_final = df_covid_clean.withColumn(
    "doenca", lit("covid")
)

display(df_covid_final)

6. Reorder final columns

In [0]:
df_covid_final = df_covid_final.select(
    "ano",
    "mes",
    "casos",
    "obitos",
    "doenca"
)

display(df_covid_final)

7. Month order

In [0]:
from pyspark.sql.functions import when, col

df_covid_final = df_covid_final.withColumn(
    "ordem_mes",
    when(col("mes") == "JANEIRO", 1)
    .when(col("mes") == "FEVEREIRO", 2)
    .when(col("mes") == "MARÇO", 3)
    .when(col("mes") == "ABRIL", 4)
    .when(col("mes") == "MAIO", 5)
    .when(col("mes") == "JUNHO", 6)
    .when(col("mes") == "JULHO", 7)
    .when(col("mes") == "AGOSTO", 8)
    .when(col("mes") == "SETEMBRO", 9)
    .when(col("mes") == "OUTUBRO", 10)
    .when(col("mes") == "NOVEMBRO", 11)
    .when(col("mes") == "DEZEMBRO", 12)
)

7. Final save

In [0]:
df_covid_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.fato_covid")

8. Validate

In [0]:
df_covid_final.count()
display(
    df_covid_final.orderBy("ano", "ordem_mes")
)
df_covid_final.printSchema()